# 2. Look inside the file

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Khoshkhah/duckOSM/blob/main/docs/notebooks/02_inspect.ipynb) <a href="https://khoshkhah.github.io/duckOSM/notebooks/02_inspect.ipynb" download><img src="https://img.shields.io/badge/Download-notebook-F37626?logo=jupyter&logoColor=white" alt="Download the notebook"></a>

<img src="https://raw.githubusercontent.com/Khoshkhah/duckOSM/main/docs/notebooks/images/banner-2.svg" alt="duckOSM notebook manual, chapter 2: Look inside the file" width="100%">

| **You learn** | the schemas and tables of a network file; the columns and their descriptions; the numbers of a table; the mode views; edges and nodes as GeoDataFrames; SQL on the file; your own DuckDB connection; what `edge_id` is |
|---|---|
| **Time** | about 2 minutes |
| **Extras** | `duckosm[gis]` (the install cell installs them) |
| **Next** | [3. Draw maps](03_maps.md): the live map, the dashboard and the static picture |

In [1]:
# Install duckOSM when it is not installed (for example in Colab). It takes approximately 1 minute.
# Until duckOSM 0.9.0 is on PyPI, the SDK of these notebooks is in the nightly wheel of main (the GitHub pre-release "nightly").
import importlib.util, json, os, urllib.request
def nightly_wheel():       # the address of the newest nightly wheel
    release = json.load(urllib.request.urlopen("https://api.github.com/repos/Khoshkhah/duckOSM/releases/tags/nightly"))
    return next(a["browser_download_url"] for a in release["assets"] if a["name"].endswith(".whl"))
if importlib.util.find_spec("duckosm") is None:      # not installed: install it with the extras of this notebook
    %pip install -q "duckosm[gis] @ {nightly_wheel()}"
import duckosm             # the SDK
if not hasattr(duckosm, "network"):                    # an old duckOSM (0.8.0 or older), without the SDK
    %pip install -q --force-reinstall --no-deps "duckosm @ {nightly_wheel()}"
    raise SystemExit("duckOSM is updated: restart the session (in Colab: Runtime > Restart session), then run this cell again")
print("duckosm", duckosm.__version__)    # the version
# The network of chapter 1 (monaco.duckdb). The chapter opens it. When you run this chapter alone (in Colab), the file
# is not there, so this cell makes it one time.
if not os.path.exists("monaco.duckdb") or not duckosm.open("monaco.duckdb").has_mode("driving", "walking", "cycling"):
    duckosm.network(osm_id=2220322, modes=["driving", "walking", "cycling"], path="monaco.duckdb")

duckosm 0.8.0


## What is in a network file

A network file is one DuckDB database. A **schema** is a group of tables in it, as a folder holds files. Each mode
that you build is one schema, with the same table names. The other schemas hold the data that all modes share.

| Schema | What it holds | The main tables |
|---|---|---|
| `driving`, `walking`, `cycling` | the network of one mode | `edges` (the roads), `nodes` (the junctions), `edge_graph` (the turns from edge to edge), `ways` and `way_nodes` (the OSM ways of the mode), `private_edges` (roads that the mode cannot use) |
| `main` | the facts of the area | `boundary` (the polygon), `global_junctions` (the nodes of all modes), `visualization_metadata` (the build time, the version, the config) |
| `raw` | the OSM data of the cut | `nodes`, `ways`, `relations` |
| `features` | the base map | `buildings`, `water_polygons`, `land`, `ocean`, `pois` and more |
| `visualization` | the drawing order | `edge_levels` (which road is drawn over which: bridges over tunnels) |
| `bus` | the bus lines | `routes`, `route_edges` |

The `driving` schema has three more tables: `turn_restrictions` (the "no left turn" rules of OSM), `lane_profile`
(the lanes) and `turn_path_restrictions`. A build option that you turn off removes its schema (chapter 1, "The build
options").

The two tables that you use most are `<mode>.edges` and `<mode>.nodes`. An **edge** is one direction of a road
between two nodes. A **node** is a point where edges meet. Each edge has a `source` node and a `target` node, so the
edges and the nodes make a directed graph.

> <img src="https://raw.githubusercontent.com/Khoshkhah/duckOSM/main/docs/notebooks/images/icon-key.svg" width="18" height="18" alt=""> **Key idea**
>
> A network file is a normal DuckDB database. Everything in this chapter is a read of its tables. You can read the same tables with SQL, with pandas or GeoPandas, or with any other DuckDB tool.

## Open the network of chapter 1

Chapter 1 builds Monaco with three modes (driving, walking and cycling) and saves it as `monaco.duckdb`. This chapter opens that file with `duckosm.open`: it builds nothing. The chapter compares the three modes.

In [2]:
net = duckosm.open("monaco.duckdb")    # the network of chapter 1: Monaco, three modes
net                                                                                               # the card

Network('monaco.duckdb', 36.7 MB, driving 2840 edges, walking 11938 edges, cycling 11305 edges)

The card names the file, its size (36.5 MB) and the build time. The table has one
row for each mode: driving has 2840 edges, walking 11938 and cycling 11305.

## The file and its modes

A `Network` is also the path of its file. `net.path` is a `pathlib.Path`, and each function that takes a file name
takes `net` too. `net.modes` lists the schemas that have a road table. `net.boundary` is the polygon of the area as a
GeoDataFrame. `net.osm_version` gives the version of the OSM data of the file: the time of the data, the Geofabrik
extract and the SHA-256 of the OSM file (the table `main.osm_source`).

In [3]:
print(net.path.name)       # the name of the file
print(net.modes)           # the modes in the file
net.boundary               # the polygon of the area, with its name, OSM id and area in km2

monaco.duckdb
['driving', 'walking', 'cycling']


,OGC_FID,name,osm_id,admin_level,area_km2,source,geometry
0,0,Monaco,2220322,None,2.39,Nominatim,"MULTIPOLYGON (((7.40903 43.72958, 7.40908 43.7..."


The boundary is one row: the relation 2220322, "Monaco", 2.39 km2, from Nominatim. The
`geometry` column is a MultiPolygon in longitude and latitude (EPSG:4326).

## The tables: `net.tables()` and `net.schema()`

`net.tables()` gives the name of each table as `schema.table`. `net.schema()` gives one row for each table, with
the number of rows, the number of columns and a **description**. The build writes each description into the file as
a DuckDB comment. The text is the same as on the reference page
[The database schema](../reference/database.md).

In [4]:
tables = net.tables()      # a list of "schema.table"
print(len(tables), "tables")
tables[:8]                 # the first eight

53 tables


['bus.route_edges',
 'bus.routes',
 'cycling.edge_graph',
 'cycling.edge_id_map',
 'cycling.edges',
 'cycling.nodes',
 'cycling.private_edges',
 'cycling.turn_permission']

In [5]:
import pandas as pd
pd.set_option("display.max_colwidth", 120)    # show more of each description
net.schema()                                  # one row for each table: rows, columns, description

,schema,table,rows,columns,description
0,bus,route_edges,5621,6,One row for each route and edge.
1,bus,routes,56,11,One row per route relation.
2,cycling,edge_graph,31366,5,"The graph of legal turns: one row for each pair of edges you can drive (or walk, or ride) one after the other. Edge-..."
3,cycling,edge_id_map,0,5,"For each merged edge, the edges that it was made from. It can be empty, when no edges merge. Columns and use: edge_i..."
4,cycling,edges,11305,30,One row per direction of a road: a two-way road has two edges. What each mode keeps: What each network contains.
5,cycling,nodes,4650,4,The ends of the routable edges.
6,cycling,private_edges,133,24,"The roads that the mode cannot use. They are drawn on the maps, but never routed and never exported."
7,cycling,turn_permission,0,7,"The turns that edge_graph cannot show: the turns of a restriction with an exception, a vehicle class or a condition...."
8,cycling,virtual_nodes,0,2,
9,cycling,way_nodes,17128,3,"The nodes of ways, one row per node."


> <img src="https://raw.githubusercontent.com/Khoshkhah/duckOSM/main/docs/notebooks/images/icon-output.svg" width="18" height="18" alt=""> **What the output shows**
>
> The file has 53 tables in eight schemas. The largest tables are the turns of walking (`walking.edge_graph`, 35074 rows) and the OSM nodes (`raw.nodes`, 34461 rows). Some tables have 0 rows: Monaco has no turn permissions and no virtual nodes, but the table is there, so the same query works on each file.

> <img src="https://raw.githubusercontent.com/Khoshkhah/duckOSM/main/docs/notebooks/images/icon-tip.svg" width="18" height="18" alt=""> **Tip**
>
> The row counts of `net.schema()` are exact: it counts each table. DuckDB's own `duckdb_tables()` gives only an estimate.

### The tables of one schema

`net.schema("driving")` gives only the tables of one schema. A wrong name gives an error that lists the schemas of
the file.

In [6]:
net.schema("main")         # the facts of the area

,schema,table,rows,columns,description
0,main,boundary,1,7,"In a build from a PBF, the boundary file as read by DuckDB: one row per feature, a column per GeoJSON property, and ..."
1,main,global_junctions,4326,2,"The nodes where every mode cuts its ways, so a way has the same edges in every mode. A node is in the table when two..."
2,main,visualization_metadata,1,9,"One row: where to open a map, and the time zone of the area."


> <img src="https://raw.githubusercontent.com/Khoshkhah/duckOSM/main/docs/notebooks/images/icon-output.svg" width="18" height="18" alt=""> **What the output shows**
>
> `main` has three tables: the boundary (1 row), the junctions of all modes (4326 rows) and the metadata of the build (1 row). The description of `global_junctions` starts with "The nodes where every mode cuts its ways". `boundary` has no description: its section of the reference page has no text before its table.

## The columns: `net.columns()`

`net.columns(table)` gives the columns of one table: the name, the DuckDB type and the description. Read it before
you write SQL or a filter, to know the names and the types.

In [7]:
net.columns("driving.edges")      # the 28 columns of the driving roads

,column_name,column_type,description
0,edge_id,BIGINT,"the edge's id: (hash(osm_id, source, target) >> 1)::BIGINT. The same on every rebuild and in every mode. Stable edge..."
1,edge_ref,VARCHAR,"a readable id, {osm_id}#{seq} then f or r for the direction (for example 41790239#2f); seq counts the way's edges fr..."
2,source,BIGINT,the node the edge starts at
3,target,BIGINT,the node the edge ends at
4,osm_id,BIGINT,the OSM way. A merged edge has the way at its source end. Negative for a connector edge
5,highway,VARCHAR,the OSM highway value
6,name,VARCHAR,the OSM name
7,oneway,BOOLEAN,TRUE when the edge has no reverse twin of the same way. Never NULL. One-way roads
8,lanes,INTEGER,"lanes in this edge's direction. Never NULL, except on connector edges (osm_id < 0). Lanes"
9,surface,VARCHAR,the OSM surface tag


> <img src="https://raw.githubusercontent.com/Khoshkhah/duckOSM/main/docs/notebooks/images/icon-output.svg" width="18" height="18" alt=""> **What the output shows**
>
> Each column has its description: for `edge_id` it is the formula `(hash(osm_id, source, target) >> 1)::BIGINT`, for `source` "the node the edge starts at". The first columns are the keys: `edge_id`, `edge_ref`, `source`, `target` and `osm_id`. Then come the OSM tags (`highway`, `name`, `oneway`, `lanes`, `surface` ...), the `geometry`, and the values that the build adds: `length_m`, `maxspeed_kmh` and `cost_s` (the travel time in seconds). The last six columns are keys of H3 cells and map tiles, for fast spatial reads.

| Column | Type | What it is |
|---|---|---|
| `edge_id` | BIGINT | the number of the edge; the same in every build and in every mode (section "What edge_id is") |
| `edge_ref` | VARCHAR | a readable name, `<osm_id>#<part>` and `f` or `r` for the direction |
| `source`, `target` | BIGINT | the node where the edge starts, and the node where it ends |
| `osm_id` | BIGINT | the OSM way of the edge |
| `highway` | VARCHAR | the road class of OSM: `primary`, `residential`, `footway` ... |
| `oneway` | BOOLEAN | `true` when the road has one direction only |
| `length_m` | FLOAT | the length in meters |
| `maxspeed_kmh` | FLOAT | the speed limit in km/h: the OSM tag, or a default for the road class |
| `cost_s` | FLOAT | the travel time in seconds at `maxspeed_kmh` |
| `geometry` | GEOMETRY | the line of the edge, in longitude and latitude |

### Compare the modes

The `edges` tables of the modes have almost the same columns. Set operations on the column names show the
differences.

In [8]:
cols = {m: set(net.columns(f"{m}.edges").column_name) for m in net.modes}   # the column names of each mode
print("walking only:", cols["walking"] - cols["driving"])
print("cycling only:", cols["cycling"] - cols["driving"])

walking only: {'walk_type'}
cycling only: {'dismount', 'cycle_type'}


Walking adds `walk_type` (for example a sidewalk or a crossing). Cycling adds
`cycle_type` and `dismount` (`true` where you must push the bicycle). Every other column is the same in the three
modes, so one query works on each mode.

## The numbers of a table: `net.summarize()`

`net.summarize(table)` runs DuckDB's `SUMMARIZE`. It gives one row for each column: the minimum, the maximum, the
approximate number of distinct values, the average, the quartiles, the count and the percentage of NULL values. The
geometry columns are left out. Use it to find the range of a column and the columns that are often empty.

In [9]:
s = net.summarize("driving.edges")                                         # one row for each column
s[["column_name", "min", "max", "approx_unique", "null_percentage"]]       # five of its columns

,column_name,min,max,approx_unique,null_percentage
0,edge_id,414242052250797,9219879820648908565,2548,0.00
1,edge_ref,1000930347#1f,992499988#1f,2114,0.00
2,source,-3945920869720410719,14358208416,1611,0.00
3,target,-3945920869720410719,14358208416,1611,0.00
4,osm_id,4097656,1551240830,1187,0.00
5,highway,living_street,unclassified,11,0.00
6,name,Accès Villa Yéyé,Virage Antony Nogues,184,32.04
7,oneway,false,true,2,0.00
8,lanes,1,3,3,0.00
9,surface,asphalt,paving_stones,3,30.00


> <img src="https://raw.githubusercontent.com/Khoshkhah/duckOSM/main/docs/notebooks/images/icon-output.svg" width="18" height="18" alt=""> **What the output shows**
>
> The driving edges are from 0.77 m to 1078 m long. The speed limits are from 10 to 70 km/h, with 7 different values. `name` is NULL for 32% of the edges (most of them are service roads), `bridge` is NULL for 99%: only 1% of the driving edges are on a bridge. `approx_unique` is an estimate: `edge_id` shows 2548, but each of the 2840 edges has its own `edge_id`.

## The view of one mode

`net.summary()` gives one row for each mode (chapter 1). `net.driving`, `net.walking` and `net.cycling` (or
`net["driving"]`) are **mode views**: a `ModeView` has the calls of the network, without `mode=`.

| Calls of a mode view | Same as |
|---|---|
| `net.walking.edges()`, `.nodes()` | `net.edges("walking")`, `net.nodes("walking")` |
| `net.walking.map()`, `.plot()`, `.dashboard()` | `net.map("walking")` ... (chapter 3) |
| `net.walking.route(a, b)` and the other routes | `net.route(a, b, mode="walking")` (chapter 7) |
| `net.walking.to_sumo(...)` and the other exports | `net.to_sumo(..., mode="walking")` (chapter 6) |
| `net.walking.summary()` | the row of walking in `net.summary()` |

In [10]:
net.summary()          # one row for each mode

,mode,edges,nodes,km,oneway_km
0,driving,2840,1768,96.2,35.5
1,walking,11938,4589,260.6,0.0
2,cycling,11305,4650,250.0,35.5


In [11]:
walk = net["walking"]     # the same as net.walking
walk.summary()            # the row of one mode

,mode,edges,nodes,km,oneway_km
0,walking,11938,4589,260.6,0.0


Walking has 11938 edges and 4589 nodes, 260.6 km in all. Driving and cycling have
35.5 km of one-way roads. Walking has 0.0 km: a person on foot can walk in both directions on each road.

> <img src="https://raw.githubusercontent.com/Khoshkhah/duckOSM/main/docs/notebooks/images/icon-warning.svg" width="18" height="18" alt=""> **Warning**
>
> A call that needs one network (a route, a graph, an export) stops with a `DuckOSMError` when the file has more modes and you give none. The message lists the modes and shows `net['driving'].route(...)`. Use a mode view, or `mode=`. `edges()`, `nodes()`, `map()` and `plot()` work on all modes at once.

## Edges and nodes as tables

`net.edges()` and `net.nodes()` read a table into a **GeoDataFrame**: a pandas DataFrame with a geometry column.
`edge_id` comes as `int64`, the float columns as `float64`, and the rows are in the order of `edge_id` (of `node_id`
for the nodes). You can then use all of pandas and GeoPandas.

| Argument | Default | What it does |
|---|---|---|
| `mode` | `None` | one mode; `None` gives the one mode of the file, or all modes with a `mode` column |
| `where` | `None` | a SQL filter on the columns, for example `"highway = 'primary'"` (only `edges`) |
| `crs` | EPSG:4326 | the coordinate system of the result: an EPSG number, or `"utm"` for the UTM zone of the area |

In [12]:
edges = net.driving.edges()                     # all driving edges
print(edges.shape, edges.crs)                   # rows and columns, the coordinate system
edges[["edge_id", "edge_ref", "osm_id", "highway", "name", "length_m", "maxspeed_kmh", "cost_s"]].head()

(2840, 28) EPSG:4326


,edge_id,edge_ref,osm_id,highway,name,length_m,maxspeed_kmh,cost_s
0,414242052250797,4230114#3r,4230114,residential,Boulevard du Ténao,44.240875,30.0,5.308905
1,2214154189574909,177052163#2f,177052163,service,None,24.298845,20.0,4.373792
2,15757195119014288,128273233#1f,128273233,residential,Rue du Ténao,9.057960,30.0,1.086955
3,16871577064511426,167625756#1f,167625756,residential,None,3.996883,30.0,0.479626
4,21502429769987032,833633729#1f,833633729,residential,None,3.554246,30.0,0.426510


2840 rows and 28 columns, in EPSG:4326 (longitude and latitude). The first rows are
the smallest `edge_id` values.

### Filter with `where`

`where` puts a SQL filter into the read. DuckDB reads only the rows that you ask for, so this is faster than a filter
in pandas on a large file.

In [13]:
primary = net.edges("driving", where="highway IN ('primary', 'primary_link')")   # the primary roads
print(len(primary), "edges,", round(primary.length_m.sum() / 1000, 1), "km")

218 edges, 12.3 km


218 edges, 12.3 km: the `primary` and `primary_link` rows of the SQL table in the
section "SQL on the file" below.

> <img src="https://raw.githubusercontent.com/Khoshkhah/duckOSM/main/docs/notebooks/images/icon-warning.svg" width="18" height="18" alt=""> **Warning**
>
> The text of `where` goes into the query as it is. Give only text that you write yourself, never text that a user of your program types.

### Meters: `crs="utm"`

The geometry in longitude and latitude has degrees, not meters. `crs="utm"` gives the edges in the UTM zone of the
area (EPSG:32632 for Monaco), so `.length` and `.area` are in meters.

In [14]:
utm = net.edges("driving", crs="utm")                        # the same edges, in meters
print(utm.crs)                                               # the UTM zone of Monaco
print(round(utm.length.sum() / 1000, 1), "km from the geometry")
print(round(utm.length_m.sum() / 1000, 1), "km from length_m")

EPSG:32632
96.3 km from the geometry
96.2 km from length_m


The geometry gives 96.3 km and `length_m` gives 96.2 km. The small difference comes
from the UTM projection: `length_m` is measured on the sphere.

### All modes in one table

With more modes in the file and no `mode`, `net.edges()` gives the edges of all modes in one GeoDataFrame, with a
`mode` column. A road that more modes can use is in it once for each mode, with the same `edge_id`.

In [15]:
all_edges = net.edges()                  # the three modes, with a "mode" column
all_edges["mode"].value_counts()

mode
walking    11938
cycling    11305
driving     2840
Name: count, dtype: int64

26083 rows: 11938 walking, 11305 cycling and 2840 driving. The table has the
columns of all modes. `walk_type` is empty for driving and cycling.

### The nodes

`net.nodes(mode)` gives the junctions of a mode: `node_id`, the point `geometry`, and the keys `h3_cell` and `tile`.
The node ids are OSM node ids. A negative id is a node that the build added (for example to split a road that comes
back to its start).

In [16]:
nodes = net.nodes("walking")    # the junctions of the walking network
print(len(nodes), "nodes")
nodes.head(3)

4589 nodes


,node_id,h3_cell,tile,geometry
0,-4477278223805530165,613499563497488383,1778020967615141660,POINT (7.4142 43.73102)
1,-4385765079388595572,613499563499585535,1778020965166870812,POINT (7.42126 43.73148)
2,-3945920869720410719,613499563587665919,1778020871309990684,POINT (7.42338 43.74088)


## SQL on the file: `net.sql()`

`net.sql(query)` runs a SQL query on the file and gives a pandas DataFrame. The connection is read-only, so a query
cannot change the file. The DuckDB extension `spatial` is loaded, so the `ST_` functions work. Use SQL for counts,
sums and joins: DuckDB computes them in the file, without a read of all the rows into Python.

In [17]:
net.sql("""
    SELECT highway, count(*) AS edges, round(sum(length_m) / 1000, 1) AS km
    FROM driving.edges
    GROUP BY highway
    ORDER BY edges DESC
""")                                                     # the road classes of driving

,highway,edges,km
0,residential,919,31.7
1,service,855,23.7
2,secondary,429,14.9
3,tertiary,387,12.5
4,primary,197,11.2
5,primary_link,21,1.1
6,secondary_link,13,0.4
7,unclassified,13,0.6
8,tertiary_link,3,0.1
9,living_street,3,0.1


Residential streets (919 edges, 31.7 km) and service roads (855 edges, 23.7 km) are
more than half of the driving network. The primary roads have 197 edges and 11.2 km.

### Values in a query: `params`

Put a value into a query with `$name` (or `?`) and `params`, not with Python text. DuckDB then reads the value as a
value, and a quote in a name cannot break the query.

In [18]:
net.sql("SELECT edge_id, edge_ref, length_m FROM driving.edges WHERE name = $name ORDER BY edge_ref",
        {"name": "Boulevard Albert 1er"})                 # the edges of one street

,edge_id,edge_ref,length_m
0,5502834370235035868,1019174508#1f,41.102180
1,4394827224625659348,1019174508#2f,4.882707
2,8440276820059948046,1019174508#3f,90.155556
3,8561693764026542227,1019174508#4f,5.178436
4,8037909646003515605,1453878835#1f,23.786472
5,2037729375342856465,161882794#1f,34.672100
6,8365478864863133107,166399479#1f,5.890634
7,1873918904400324117,166399479#2f,87.658560
8,3622281415122805437,166399479#3f,15.597761
9,3325892313673521836,166399479#4f,3.941422


### Spatial SQL

The spatial functions take the geometry columns. Here the area of the boundary in km2, measured in the UTM zone, and
the number of driving edges within 200 m of the casino square.

In [19]:
net.sql("""
    SELECT round(ST_Area(ST_Transform(geom, 'EPSG:4326', 'EPSG:32632', always_xy := true)) / 1e6, 2) AS km2
    FROM main.boundary
""")                                                     # the area of Monaco

,km2
0,2.39


In [20]:
net.sql("""
    SELECT count(*) AS edges
    FROM driving.edges
    WHERE ST_DWithin(ST_Transform(geometry, 'EPSG:4326', 'EPSG:32632', always_xy := true),
                     ST_Transform(ST_Point(7.4280, 43.7393), 'EPSG:4326', 'EPSG:32632', always_xy := true), 200)
""")                                                     # the edges near the casino square

,edges
0,223


The boundary has 2.39 km2, the same value as the `area_km2` column of
`net.boundary`. 223 driving edges are within 200 m of the casino square.

> <img src="https://raw.githubusercontent.com/Khoshkhah/duckOSM/main/docs/notebooks/images/icon-tip.svg" width="18" height="18" alt=""> **Tip**
>
> `always_xy := true` tells DuckDB that the points are longitude first. Without it, `ST_Transform` reads EPSG:4326 as latitude first.

## Your own DuckDB connection: `net.connect()`

`net.sql()` opens a connection for each query and gives pandas. `net.connect()` gives a read-only DuckDB connection
that stays open, with `spatial` loaded. Use it for many queries in a row, for DuckDB relations, for Arrow
(`.arrow()`) or for Polars (`.pl()`). Close it after use: a `with` block closes it for you.

In [21]:
with net.connect() as con:                                          # closed at the end of the block
    rel = con.sql("SELECT edge_id, highway, length_m FROM walking.edges")    # a DuckDB relation: not read yet
    print(rel.count("*").fetchone()[0], "walking edges")                     # DuckDB counts in the file
    print(rel.filter("highway = 'steps'").count("*").fetchone()[0], "of them are steps")
    table = rel.limit(5).to_arrow_table()                                    # an Arrow table, no pandas
print(type(table).__name__, table.num_rows, "rows")

11938 walking edges
1038 of them are steps
Table 5 rows


11938 walking edges, and 1038 of them are steps. The Arrow table has 5 rows. The
relation reads nothing until you ask for a result. The two counts run in
DuckDB, and only the numbers come to Python. `to_arrow_table()` gives a pyarrow `Table` for tools that read Arrow.

> <img src="https://raw.githubusercontent.com/Khoshkhah/duckOSM/main/docs/notebooks/images/icon-warning.svg" width="18" height="18" alt=""> **Warning**
>
> In DuckDB 1.5, `.arrow()` gives a `RecordBatchReader`, which reads the rows only when you use it. Read it to the end (`.read_all()`) before the connection closes. A reader that you do not read keeps the query open, and the next query on the file can wait for it. `to_arrow_table()` reads all rows at once.

> <img src="https://raw.githubusercontent.com/Khoshkhah/duckOSM/main/docs/notebooks/images/icon-tip.svg" width="18" height="18" alt=""> **Tip**
>
> You can also open the file with `duckdb.connect(str(net.path), read_only=True)`. Then load the extension yourself: `con.load_extension("spatial")`.

## What `edge_id` is

`edge_id` is the key of an edge. It is a number of type `BIGINT` (64 bits). The build computes it from three values
of the edge: the OSM way (`osm_id`) and the two nodes (`source`, `target`).

<img src="https://raw.githubusercontent.com/Khoshkhah/duckOSM/main/docs/notebooks/images/edge-id.svg" alt="A two-way road is two edges: each direction has its own edge_ref and edge_id" width="100%">

| Property | Why it is important |
|---|---|
| It comes from the OSM data, not from the row number | a new build of the same OSM data gives the same `edge_id` to each edge |
| Each direction has its own number | `source` and `target` change places, so the two edges of a two-way road get different numbers |
| It is the same in each mode | a street with a sidewalk has the same `edge_id` in `driving.edges` and in `walking.edges` |
| It is a `BIGINT` | keep it as `int64` in pandas; a `float64` loses the last digits (chapter 4) |

`edge_ref` is a second name of an edge that people can read. It can change when a way gets more nodes in OSM, so join
your data on `edge_id`, not on `edge_ref`.

> <img src="https://raw.githubusercontent.com/Khoshkhah/duckOSM/main/docs/notebooks/images/icon-key.svg" width="18" height="18" alt=""> **Key idea**
>
> `edge_id` is the key between duckOSM and your data. It stays the same when you build the area again and it is the same in each mode, so data that you join on it stays correct.

### Two edges for a two-way road

The OSM way 4230114 is a two-way street. Each part of the way between two nodes gives two edges: `f` in the drawing
direction of the way, `r` against it.

In [22]:
net.sql("""
    SELECT edge_id, edge_ref, source, target, is_reverse
    FROM driving.edges WHERE osm_id = 4230114 ORDER BY edge_ref
""")                                                     # the edges of one way

,edge_id,edge_ref,source,target,is_reverse
0,925170278510060820,4230114#1f,21921299,3625063392,False
1,7518368909814682313,4230114#1r,3625063392,21921299,True
2,3093664370798973039,4230114#2f,3625063392,1080509801,False
3,8638126748304063112,4230114#2r,1080509801,3625063392,True
4,4950030941241418106,4230114#3f,1080509801,1868723946,False
5,414242052250797,4230114#3r,1868723946,1080509801,True


Three parts of the way give six edges. The two edges of each part have the same
nodes in the other order, and two very different numbers: `4230114#3f` is 4950030941241418106, `4230114#3r` is
414242052250797. `is_reverse` is `true` for the `r` edges.

### Compute an `edge_id`

`duckosm.edge_id_hash(osm_id, source, target)` computes the `edge_id` in Python. The same formula is in each file as
the SQL macro `edge_id_hash`. `duckosm.edge_id_expr()` gives the SQL text of the formula, for your own queries.

In [23]:
row = edges.iloc[0]                                                    # one driving edge
print(row.edge_id, duckosm.edge_id_hash(row.osm_id, row.source, row.target))   # the same number
print(duckosm.edge_id_expr())                                          # the formula as SQL

414242052250797 414242052250797
(hash(osm_id::BIGINT, source::BIGINT, target::BIGINT) >> 1)::BIGINT


In [24]:
net.sql("SELECT count(*) AS different FROM driving.edges WHERE edge_id <> edge_id_hash(osm_id, source, target)")

,different
0,0


The Python function gives the same number as the column. The SQL macro agrees for
all 2840 driving edges: 0 are different.

| Call | Gives |
|---|---|
| `duckosm.edge_id_hash(osm_id, source, target)` | the `edge_id` of one edge |
| `duckosm.edge_id_expr("osm_id", "source", "target")` | the SQL text of the formula, with your column names |
| `duckosm.edge_id_hash_v1(...)`, `duckosm.edge_id_expr_v1(...)` | the previous formula, with `is_reverse`: to connect old ids to new ids |
| `duckosm.create_edge_id_macro(con)` | writes the two SQL macros into a DuckDB database of yours |

> <img src="https://raw.githubusercontent.com/Khoshkhah/duckOSM/main/docs/notebooks/images/icon-warning.svg" width="18" height="18" alt=""> **Warning**
>
> The formula uses DuckDB's `hash()`. Another DuckDB major version can give other numbers. duckOSM pins DuckDB 1.x for this reason. Outside DuckDB, join on the three columns `osm_id`, `source` and `target`.

### The same edge in each mode

A road that more modes can use has the same `edge_id` in each mode. A join of two `edges` tables on `edge_id` gives
the roads that the two modes share.

In [25]:
net.sql("""
    SELECT 'driving and walking' AS modes, count(*) AS edges FROM driving.edges JOIN walking.edges USING (edge_id)
    UNION ALL
    SELECT 'driving and cycling', count(*) FROM driving.edges JOIN cycling.edges USING (edge_id)
    UNION ALL
    SELECT 'all modes, each road once', count(DISTINCT edge_id)
    FROM (SELECT edge_id FROM driving.edges UNION ALL SELECT edge_id FROM walking.edges
          UNION ALL SELECT edge_id FROM cycling.edges)
""")                                                     # the roads that the modes share

,modes,edges
0,driving and walking,2201
1,driving and cycling,2829
2,"all modes, each road once",13302


2201 of the 2840 driving edges are also walking edges, and 2829 are also cycling
edges. The three modes have 13302 different edges in all, less than the 26083 rows of the three tables, because most
roads are in two or three modes.

## Try it yourself

**1.** Find the five longest walking edges that have a name. Use `net.sql`.

<details><summary>Show the answer</summary>

```python
net.sql("SELECT name, highway, round(length_m) AS m FROM walking.edges WHERE name IS NOT NULL ORDER BY length_m DESC LIMIT 5")
```
</details>

**2.** Which columns of `walking.edges` are NULL for more than 90% of the rows?

<details><summary>Show the answer</summary>

```python
s = net.summarize("walking.edges")
s[s.null_percentage > 90].column_name.tolist()
```
</details>

**3.** Get the steps of the walking network as a GeoDataFrame in meters, and give their total length in km.

<details><summary>Show the answer</summary>

```python
steps = net.walking.edges(where="highway = 'steps'", crs="utm")
round(steps.length.sum() / 1000, 1)
```
</details>

**4.** Count the driving edges that are not in the walking network, by road class.

<details><summary>Show the answer</summary>

```python
net.sql("""SELECT highway, count(*) AS n FROM driving.edges
           WHERE edge_id NOT IN (SELECT edge_id FROM walking.edges)
           GROUP BY highway ORDER BY n DESC""")
```
</details>

## Words in this chapter

| Word | Meaning |
|---|---|
| schema | a group of tables in a DuckDB file: one for each mode, and `main`, `raw`, `features` ... |
| table | rows and columns in a schema, named `schema.table`, for example `driving.edges` |
| description | the text of a table or a column, written into the file as a DuckDB comment |
| `ModeView` | the view of one mode: `net.walking`, with the calls of the network and no `mode=` |
| GeoDataFrame | a pandas DataFrame with a geometry column (GeoPandas) |
| CRS | the coordinate system of a geometry: EPSG:4326 (degrees) or a UTM zone (meters) |
| `edge_ref` | the readable name of an edge, `<osm_id>#<part>f` or `r`; it can change |
| `edge_id` | the key of an edge: a hash of `osm_id`, `source` and `target`; the same in each build and mode |
| relation (DuckDB) | a query that DuckDB runs only when you ask for its result |

## Summary

| Call | Does |
|---|---|
| `net.path`, `net.modes`, `net.boundary` | the file, the modes, the polygon of the area |
| `net.tables()` | the tables, as `schema.table` |
| `net.schema(schema=None)` | one row for each table: rows, columns, description |
| `net.columns(table)` | the columns of a table: name, type, description |
| `net.summarize(table)` | DuckDB's `SUMMARIZE`: the range, the distinct values and the NULLs of each column |
| `net.summary()`, `net.walking`, `net["walking"]` | one row for each mode; the view of one mode |
| `net.edges(mode, where=, crs=)` | the edges as a GeoDataFrame |
| `net.nodes(mode, crs=)` | the nodes as a GeoDataFrame |
| `net.sql(query, params=None)` | a SQL query on the file, as a DataFrame |
| `net.connect()` | a read-only DuckDB connection with `spatial` |
| `duckosm.edge_id_hash(...)`, `duckosm.edge_id_expr()` | the formula of `edge_id`, in Python and as SQL |

The reference: [Python SDK, What is in the file](../reference/python.md#what-is-in-the-file) and [The database schema](../reference/database.md). **Next:** [3. Draw maps](03_maps.md), the live map, the dashboard and the static picture.